# Elicitation: ask the user a question

Goal: park a run on a human question with `ElicitationToolset`, answer it
with `resolve_interaction`, and watch the answer come back to the model as
the tool result.

Trust: [T2](../../docs/site/security-trust-levels.md) callback. Network: none.

The elicitation toolset exposes tools that never complete inline. A call
builds a durable `InteractionRequest`, the runtime journals it and moves the
run to its awaiting-interaction phase, and `run.list_interactions()` surfaces
the pending question. When a person resolves it, the runtime re-invokes the
tool with the resolution merged into the original arguments and the tool
returns `{"answer": ...}` as its completed result — so the model reads the
human's answer exactly where it expects the tool output.

## Build the toolset

`ask_user=True` exposes the free-form `ask_user` tool: the model writes the
prompt and picks a profile — `free_text` (default), `choice` with
`options`, or `form` with an inline JSON schema. `tools=[...]` registers
typed per-workflow tools whose prompt and response schema are fixed here at
registration; the model can only add call-time `context`, never reshape the
question contract.

In [ ]:
import finstack_ai

elicitation = finstack_ai.ElicitationToolset(
    ask_user=True,
    tools=[
        {
            "name": "confirm_trade_params",
            "title": "Confirm trade parameters",
            "description": "Ask the operator to confirm trade parameters before execution.",
            "prompt": "Please confirm the trade parameters.",
            "kind": "form",
            "response_schema": {
                "type": "object",
                "properties": {"confirmed": {"type": "boolean"}},
                "required": ["confirmed"],
            },
        }
    ],
)
print(elicitation.component, elicitation.tool_count)
assert elicitation.tool_count == 2

## A scripted model that asks

The first model turn calls `ask_user`. The second turn runs only after the
question is answered; it captures the request so we can inspect what the
model saw.

In [ ]:
from typing import Any

model_calls = 0
resumed_request: dict[str, Any] = {}


async def scripted_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context
    global model_calls
    model_calls += 1
    if model_calls == 1:
        return {
            "text": "",
            "completion_id": "elicit-1",
            "tool_calls": [
                {
                    "name": "ask_user",
                    "arguments": {"prompt": "What is the maximum position size?"},
                }
            ],
        }
    resumed_request.update(request)
    return {"text": "Position capped as instructed.", "completion_id": "elicit-2"}


agent = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        scripted_model,
        component="notebook.model.elicitation",
        provider="notebook",
        model="notebook-model",
    ),
    [elicitation],
    "Ask the user when information is missing.",
)

## Start the run and read the pending question

`agent.start` returns immediately. The run parks once `ask_user` executes;
poll `list_interactions()` until the question appears.

In [ ]:
import asyncio

run = agent.start("Size the new position.")

pending: dict[str, Any] = {}
for _ in range(300):
    interactions = await run.list_interactions()
    if interactions:
        pending = interactions[0]
        break
    await asyncio.sleep(0.01)

assert pending, "expected a pending interaction"
print(pending["kind"])
print(pending["prompt"])
print(pending["response_schema"])
assert pending["kind"] == {"kind": "free_text"}

## Answer it

A resolution names the interaction, the resolving principal, and a response
matching the request's schema. Free-text answers live under the reserved
`answer` key. The run resumes on its own; the resolved answer becomes the
`ask_user` tool result.

In [ ]:
resolution = {
    "interaction_id": pending["interaction_id"],
    "resolution_id": "notebook-resolution-1",
    "principal": {
        "issuer": "finstack-ai-python",
        "subject": "local-user",
        "tenant_scope": "python-local",
    },
    "authorization": {
        "policy_version": "python-policy-v1",
        "decision_id": "python-decision-v1",
    },
    "response": {"answer": "250k USD"},
}
await run.resolve_interaction(resolution)

result = await run.result()
print(result.text)
assert result.text == "Position capped as instructed."
assert model_calls == 2

The second model request now carries the human's answer as the tool
result of the `ask_user` call:

In [ ]:
for message in resumed_request["messages"]:
    if message["role"] == "tool":
        print(message["content"])

import json

assert "250k USD" in json.dumps(resumed_request["messages"])

## Typed per-workflow elicitation

`confirm_trade_params` was registered with a fixed prompt and response
schema. The model supplies only `context`; the registered schema is nested
under `answer`, so a form resolution answers with
`{"answer": {"confirmed": true}}`.

In [ ]:
typed_calls = 0


async def typed_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context, request
    global typed_calls
    typed_calls += 1
    if typed_calls == 1:
        return {
            "text": "",
            "completion_id": "typed-1",
            "tool_calls": [
                {
                    "name": "confirm_trade_params",
                    "arguments": {"context": "Buy 100 AAPL @ market."},
                }
            ],
        }
    return {"text": "Trade confirmed and queued.", "completion_id": "typed-2"}


typed_agent = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        typed_model,
        component="notebook.model.typed-elicitation",
        provider="notebook",
        model="notebook-model",
    ),
    [elicitation],
    "Confirm before executing.",
)

typed_run = typed_agent.start("Buy 100 AAPL.")
typed_pending: dict[str, Any] = {}
for _ in range(300):
    interactions = await typed_run.list_interactions()
    if interactions:
        typed_pending = interactions[0]
        break
    await asyncio.sleep(0.01)

assert typed_pending["kind"] == {"kind": "form"}
print(typed_pending["prompt"])

await typed_run.resolve_interaction(
    {
        "interaction_id": typed_pending["interaction_id"],
        "resolution_id": "notebook-resolution-2",
        "principal": {
            "issuer": "finstack-ai-python",
            "subject": "local-user",
            "tenant_scope": "python-local",
        },
        "authorization": {
            "policy_version": "python-policy-v1",
            "decision_id": "python-decision-v1",
        },
        "response": {"answer": {"confirmed": True}},
    }
)
typed_result = await typed_run.result()
print(typed_result.text)
assert typed_result.text == "Trade confirmed and queued." 

## Where to go next

- Approval-gated tools (`approval: {"requirement": "required"}` on a tool
  spec) park the same way with an approval profile; see
  `04_runs_events_sessions.ipynb` for run and event plumbing.
- Interactions are durable: with a SQLite store the pending question
  survives a process restart (`examples/durable-interaction` shows this in
  Rust).
- The server surface resolves the same interactions over the session
  protocol via the `interaction_resolved` command.